# NB_patient_360

Builds `adb_healthcare.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the STTM workbook `sttm_silver_to_gold.xlsx`. Full rebuild via CREATE OR REPLACE TABLE, LEFT JOINs from the driving patients table.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.patient_360 AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM adb_healthcare.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE) AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT) AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT) AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT) AS critical_test_count,
  l.last_test_date
FROM adb_healthcare.silver.patients p
LEFT JOIN adb_healthcare.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_healthcare.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
%sql
WITH expected AS (
  SELECT * FROM VALUES
    (1,'patient_id','int'),(2,'first_name','string'),(3,'last_name','string'),(4,'date_of_birth','date'),
    (5,'age','int'),(6,'gender','string'),(7,'city','string'),(8,'state','string'),
    (9,'hospital_id','int'),(10,'hospital_name','string'),(11,'insurance_provider_id','int'),
    (12,'insurance_provider_name','string'),(13,'total_tests','int'),(14,'abnormal_test_count','int'),
    (15,'critical_test_count','int'),(16,'last_test_date','date')
  AS t(position, column_name, expected_type)
),
actual AS (
  SELECT ordinal_position + 1 AS position, column_name, lower(data_type) AS actual_type
  FROM adb_healthcare.information_schema.columns
  WHERE table_schema = 'gold' AND table_name = 'patient_360'
)
SELECT
  COALESCE(e.position, a.position) AS position,
  COALESCE(e.column_name, a.column_name) AS column_name,
  e.expected_type,
  a.actual_type,
  CASE WHEN e.column_name IS NULL THEN 'UNEXPECTED COLUMN'
       WHEN a.column_name IS NULL THEN 'MISSING'
       WHEN e.expected_type <> a.actual_type THEN 'TYPE MISMATCH'
       WHEN e.position <> a.position THEN 'ORDER MISMATCH'
       ELSE 'OK' END AS status
FROM expected e
FULL OUTER JOIN actual a ON e.column_name = a.column_name
ORDER BY position;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM adb_healthcare.silver.patients) AS silver_patients,
  COUNT(*) AS gold_rows,
  COUNT(DISTINCT patient_id) AS distinct_patient_ids,
  COUNT_IF(hospital_name IS NULL) AS missing_hospital_name,
  COUNT_IF(insurance_provider_name IS NULL) AS missing_insurance_name,
  COUNT_IF(total_tests = 0) AS patients_without_tests,
  SUM(total_tests) AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_healthcare.silver.lab_results
    WHERE patient_id IN (SELECT patient_id FROM adb_healthcare.silver.patients)) AS silver_tests_for_known_patients,
  SUM(abnormal_test_count) AS gold_abnormal,
  SUM(critical_test_count) AS gold_critical,
  MIN(age) AS min_age,
  MAX(age) AS max_age
FROM adb_healthcare.gold.patient_360;

In [0]:
%sql
SELECT *
FROM adb_healthcare.gold.patient_360
ORDER BY total_tests DESC, patient_id
LIMIT 10;

In [0]:
# DBTITLE 1,Return patient 360 record count
updated_df = spark.table("adb_healthcare.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))